In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/jobs.csv")
sample = df.dropna(subset=["title", "description"]).sample(2000, random_state=42).reset_index(drop=True)
job_embeddings = np.load("../data/job_embeddings.npy")

print(sample.shape)
print(job_embeddings.shape)
print(sample[["title"]].head())

(2000, 31)
(2000, 384)
                                               title
0                                          Warehouse
1                 Influencer Talent Manager - Beauty
2                      Talent Attraction Specialist 
3           Retail Sales - Shoes - Stonebriar Centre
4  Physician Assistant, Per Diem - Internal Medicine


In [3]:
resume = """
Computer Science graduate with practical training and project-based experience
in Data Analytics, Python programming, SQL, and Business Intelligence.

Education:
Bachelor of Science in Computer Science (BSCS).
IBM Data Analyst Professional Certificate - 11 courses completed.
NAVTTC Data Analytics Blended Programme.

Technical Skills:
Python, SQL, PostgreSQL, Excel, Advanced Excel, Google Sheets,
Power BI, Tableau, pandas, NumPy, Matplotlib, Seaborn,
Jupyter Notebook, Data Cleaning, Data Transformation,
Exploratory Data Analysis (EDA), Data Visualization,
Dashboard Development, Data Reporting, Data Modeling,
Data Interpretation, Statistical Analysis, Git and GitHub.

Practical Projects:
1. Urban Basket Retail Sales Analysis:
Analyzed retail sales data using PostgreSQL, Python, and Power BI.
Developed business performance measures, sales KPIs,
customer analysis, and interactive dashboards.

MetroCare Hospital Analytics:
Performed data cleaning and transformation using Power Query.
Analyzed hospital service revenue, patient records,
appointments, and billing data.
Developed a Power BI dashboard.

SkyLink Airlines SQL Analysis:
Performed SQL-based analysis of airline bookings,
revenue, cancellations, and business performance.

Career Objective:
Seeking entry-level opportunities as a Data Analyst,
Junior Data Analyst, Business Intelligence Analyst,
SQL Analyst, or Python Data Analyst.

Interested in transforming raw data into meaningful
business insights and supporting data-driven decisions.
"""

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

chunks = [c.strip() for c in resume.split("\n\n") if c.strip()]
resume_vec = model.encode(chunks).mean(axis=0, keepdims=True)

scores = cosine_similarity(resume_vec, job_embeddings)[0]
top_idx = scores.argsort()[::-1][:10]

for i in top_idx:
    print(round(scores[i], 3), sample.loc[i, "title"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.71 Business Data Analyst
0.673 Data Analyst-WA
0.638 Financial Analyst
0.634 Data Analyst
0.623 Data Scientist II
0.62 Data Engineer
0.619 Tableau Architect
0.611 Data Analyst (Korean)
0.607 Healthcare Economics Analyst
0.602 Business Analyst


In [5]:
import re

skills = [
    "sql", "python", "excel", "tableau", "power bi", "looker", "qlik", "alteryx",
    "sas", "spark", "hadoop", "aws", "azure", "gcp", "snowflake", "bigquery",
    "redshift", "databricks", "airflow", "dbt", "etl", "data warehouse",
    "machine learning", "statistics", "a/b testing", "vba", "git", "docker",
    "java", "scala", "jira", "scikit-learn", "tensorflow", "power query",
]

def has_skill(text, skill):
    return re.search(r"\b" + re.escape(skill) + r"\b", text.lower()) is not None

top_jobs = sample.loc[top_idx]
rows = []
for skill in skills:
    count = sum(has_skill(d, skill) for d in top_jobs["description"])
    rows.append((skill, count, has_skill(resume, skill)))

print("Skills in the top 10 jobs that are NOT in your resume:\n")
for skill, count, in_resume in sorted(rows, key=lambda r: -r[1]):
    if count > 0 and not in_resume:
        print(f"{skill}: {count} of 10 jobs")

Skills in the top 10 jobs that are NOT in your resume:

statistics: 3 of 10 jobs
hadoop: 2 of 10 jobs
etl: 2 of 10 jobs
sas: 1 of 10 jobs
machine learning: 1 of 10 jobs
vba: 1 of 10 jobs


In [6]:
top_idx = scores.argsort()[::-1][:50]
top_jobs = sample.loc[top_idx]

skills = [s for s in skills if s != "statistics"] + ["statistic", "statistical"]

rows = []
for skill in skills:
    count = sum(has_skill(d, skill) for d in top_jobs["description"])
    rows.append((skill, count, has_skill(resume, skill)))

print("Skills in the top 50 jobs that are NOT in your resume:\n")
for skill, count, in_resume in sorted(rows, key=lambda r: -r[1]):
    if count > 0 and not in_resume:
        print(f"{skill}: {count} of 50 jobs")

Skills in the top 50 jobs that are NOT in your resume:

aws: 9 of 50 jobs
spark: 7 of 50 jobs
etl: 7 of 50 jobs
machine learning: 7 of 50 jobs
java: 7 of 50 jobs
scala: 7 of 50 jobs
azure: 6 of 50 jobs
hadoop: 5 of 50 jobs
snowflake: 4 of 50 jobs
sas: 3 of 50 jobs
redshift: 3 of 50 jobs
databricks: 3 of 50 jobs
airflow: 2 of 50 jobs
vba: 2 of 50 jobs
dbt: 1 of 50 jobs
data warehouse: 1 of 50 jobs


## Stage 4 Findings

- Rebuilt the same 2,000-posting sample and matched my resume to the top 10 and top 50 jobs using sentence embeddings.
- Counted which common data skills appear in those job descriptions but not in my resume.
- No missing skill appears in more than 18% of the top 50 jobs (AWS was highest at 9 of 50), so my resume already covers the core analyst skills.
- Spark, Scala, Java and Hadoop come from data engineer roles, so they are less relevant to my target roles.
- ETL is worded differently from my Power Query transformation work, so I can add the term to my resume.
- Cloud platforms (AWS, Azure, Snowflake) are the real gaps. Snowflake is the best first step because it builds on my SQL skills.
- Limitation: a 2,000-posting sample, simple keyword matching, and "mentioned in the job" does not mean "required".